# RCNN Object Detection - Pascal VOC 2012

Klasszikus RCNN (Girshick et al. 2014) implementáció PyTorch és scikit-learn használatával.
ResNet50 backbone, Selective Search régió javaslatok, SVM osztályozás.

## 0. Környezet beállítása

In [ ]:
# GPU ellenőrzés
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA elérhető: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Függőségek telepítése
!pip install -q kagglehub opencv-python scikit-learn matplotlib pillow torch torchvision

import kagglehub
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.svm import LinearSVC
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import xml.etree.ElementTree as ET
import os
import random
import pickle
from collections import defaultdict
from pathlib import Path

print("Minden függőség betöltve.")

## 1. Adathalmaz feltérképezése

Pascal VOC 2012 letöltése kagglehub segítségével, osztályeloszlás, képméretek és bbox statisztikák vizsgálata.

In [ ]:
# Pascal VOC 2012 letöltése
path = kagglehub.dataset_download("gopalbhattrai/pascal-voc-2012-dataset")
print(f"Adathalmaz elérési út: {path}")

# Mappaszerkezet feltárása
voc_root = Path(path) / "VOC2012"
annotations_dir = voc_root / "Annotations"
images_dir = voc_root / "JPEGImages"
imgset_dir = voc_root / "ImageSets" / "Main"

print(f"Annotations: {annotations_dir.exists()}")
print(f"Images: {images_dir.exists()}")

# Képek listája train/val splitből
train_images = []
for class_file in sorted(imgset_dir.glob("*_train.txt")):
    with open(class_file) as f:
        train_images.extend([line.strip().split()[0] for line in f if line.strip()])
train_images = list(set(train_images))

val_images = []
for class_file in sorted(imgset_dir.glob("*_val.txt")):
    with open(class_file) as f:
        val_images.extend([line.strip().split()[0] for line in f if line.strip()])
val_images = list(set(val_images))

print(f"Train képek: {len(train_images)}")
print(f"Val képek: {len(val_images)}")

In [ ]:
# Osztályok listája
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def parse_voc_xml(xml_path):
    """VOC XML annotáció parse-olása. Visszaadja: [(osztály, [xmin,ymin,xmax,ymax]), ...]"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append((name, [xmin, ymin, xmax, ymax]))
    return objects, (width, height)

# Osztályeloszlás vizsgálata
class_counts = defaultdict(int)
bbox_areas = []
bbox_ratios = []
image_sizes = []

for img_id in train_images[:500]:
    xml_path = annotations_dir / f"{img_id}.xml"
    if xml_path.exists():
        objects, (w, h) = parse_voc_xml(xml_path)
        image_sizes.append((w, h))
        for cls, bbox in objects:
            class_counts[cls] += 1
            area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            ratio = (bbox[2] - bbox[0]) / max(bbox[3] - bbox[1], 1)
            bbox_areas.append(area)
            bbox_ratios.append(ratio)

print("=== Osztályeloszlás (első 500 train kép) ===")
for cls in VOC_CLASSES:
    print(f"  {cls}: {class_counts[cls]}")

print(f"\nBbox terület: min={min(bbox_areas):.0f}, max={max(bbox_areas):.0f}, mean={np.mean(bbox_areas):.0f}")
print(f"Bbox arány (w/h): mean={np.mean(bbox_ratios):.2f}")
print(f"Képméret: min={min(image_sizes)}, max={max(image_sizes)}")

In [ ]:
# Vizualizáció: minta képek annotációkkal és statisztikák

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 3 minta kép bboxokkal
sample_ids = random.sample(train_images[:500], 3)
for i, img_id in enumerate(sample_ids):
    img_path = images_dir / f"{img_id}.jpg"
    img = Image.open(img_path)
    xml_path = annotations_dir / f"{img_id}.xml"
    objects, _ = parse_voc_xml(xml_path)

    axes[0, i].imshow(img)
    for cls, bbox in objects:
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                                   linewidth=2, edgecolor='red', facecolor='none')
        axes[0, i].add_patch(rect)
        axes[0, i].text(bbox[0], bbox[1]-5, cls, color='red', fontsize=8)
    axes[0, i].set_title(f"{img_id} ({len(objects)} objektum)")
    axes[0, i].axis('off')

# Osztályeloszlás hisztogram
axes[1, 0].bar(range(len(VOC_CLASSES)), [class_counts[c] for c in VOC_CLASSES])
axes[1, 0].set_xticks(range(len(VOC_CLASSES)))
axes[1, 0].set_xticklabels(VOC_CLASSES, rotation=90, fontsize=7)
axes[1, 0].set_title("Osztályeloszlás")

# Bbox terület hisztogram
axes[1, 1].hist(bbox_areas, bins=50, edgecolor='black')
axes[1, 1].set_title("Bbox területek eloszlása")
axes[1, 1].set_xlabel("Terület (px²)")

# Bbox arány hisztogram
axes[1, 2].hist(bbox_ratios, bins=50, edgecolor='black')
axes[1, 2].set_title("Bbox szélesség/magasság arány")
axes[1, 2].set_xlabel("w/h arány")

plt.tight_layout()
plt.savefig('dataset_exploration.png', dpi=100)
plt.show()
print("Adathalmaz feltérképezés kész.")

## 2. RCNN háló létrehozása

ResNet50 backbone ImageNet előtanítással, régiók átméretezése 224x224-re, SVM osztályozó és Ridge regresszor osztályonként.

In [ ]:
# Feature extractor: ResNet50 utolsó osztályozó réteg nélkül
class FeatureExtractor(nn.Module):
    """ResNet50 backbone, az avgpool előtti feature map-eket adja vissza."""
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.output_dim = 2048

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

# Eszköz beállítása
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
feature_extractor = FeatureExtractor().to(device)
feature_extractor.eval()
print(f"Feature extractor létrehozva. Kimeneti dimenzió: {feature_extractor.output_dim}")
print(f"Eszköz: {device}")

In [ ]:
# Kép transzformáció a ResNet-hez
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_region_features(image, regions, batch_size=64):
    """Régiók kivágása, átméretezése és feature kinyerése batch-elve."""
    features_list = []
    for i in range(0, len(regions), batch_size):
        batch_regions = regions[i:i+batch_size]
        batch_tensors = []
        for (x, y, w, h) in batch_regions:
            crop = image.crop((x, y, x+w, y+h))
            crop_tensor = transform(crop)
            batch_tensors.append(crop_tensor)
        batch = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            feats = feature_extractor(batch)
        features_list.append(feats.cpu().numpy())
    return np.vstack(features_list) if features_list else np.array([])

print("Feature kinyerő függvények definiálva.")

In [ ]:
class RCNN(nn.Module):
    """Klasszikus RCNN modell: backbone + SVM osztályozók + Ridge regresszorok."""
    def __init__(self, num_classes=20):
        super().__init__()
        self.num_classes = num_classes
        self.feature_extractor = FeatureExtractor()
        self.svm_classifiers = {}
        self.ridge_regressors = {}
        self.scaler = StandardScaler()
        self.class_names = VOC_CLASSES

    def extract_features(self, image, regions, batch_size=64):
        return extract_region_features(image, regions, batch_size)

    def fit_svm(self, features, labels):
        """SVM osztályozók tanítása osztályonként (one-vs-rest)."""
        features_scaled = self.scaler.fit_transform(features)
        for cls_idx in range(self.num_classes):
            y = (labels == cls_idx).astype(int)
            if y.sum() == 0:
                continue
            svm = LinearSVC(C=0.1, max_iter=5000, dual='auto', random_state=42)
            svm.fit(features_scaled, y)
            self.svm_classifiers[cls_idx] = svm
        print(f"SVM-ek tanítva {len(self.svm_classifiers)} osztályra.")

    def fit_regressors(self, features, labels, gt_boxes, proposal_boxes):
        """Bbox regresszorok tanítása osztályonként."""
        features_scaled = self.scaler.transform(features)
        for cls_idx in range(self.num_classes):
            mask = labels == cls_idx
            if mask.sum() < 10:
                continue
            X = features_scaled[mask]
            targets = []
            for j in np.where(mask)[0]:
                gt = gt_boxes[j]
                prop = proposal_boxes[j]
                tx = (gt[0] - prop[0]) / prop[2]
                ty = (gt[1] - prop[1]) / prop[3]
                tw = np.log(max(gt[2], 1e-6) / max(prop[2], 1e-6))
                th = np.log(max(gt[3], 1e-6) / max(prop[3], 1e-6))
                targets.append([tx, ty, tw, th])
            targets = np.array(targets)
            ridge = Ridge(alpha=1000)
            ridge.fit(X, targets)
            self.ridge_regressors[cls_idx] = ridge
        print(f"Regresszorok tanítva {len(self.ridge_regressors)} osztályra.")

    def predict(self, image, regions):
        """Predikció: osztály + bbox finomítás minden régióra."""
        if len(regions) == 0:
            return [], [], []
        feats = self.extract_features(image, regions)
        feats_scaled = self.scaler.transform(feats)
        scores = np.zeros((len(regions), self.num_classes))
        refined_boxes = []
        for cls_idx in range(self.num_classes):
            if cls_idx in self.svm_classifiers:
                decision = self.svm_classifiers[cls_idx].decision_function(feats_scaled)
                scores[:, cls_idx] = decision
                if cls_idx in self.ridge_regressors:
                    deltas = self.ridge_regressors[cls_idx].predict(feats_scaled)
                    for j, (x, y, w, h) in enumerate(regions):
                        tx, ty, tw, th = deltas[j]
                        cx = x + w/2
                        cy = y + h/2
                        new_cx = cx + tx * w
                        new_cy = cy + ty * h
                        new_w = w * np.exp(tw)
                        new_h = h * np.exp(th)
                        refined_boxes.append((new_cx - new_w/2, new_cy - new_h/2,
                                              new_cx + new_w/2, new_cy + new_h/2))
        return scores, refined_boxes

print("RCNN osztály definiálva.")

## 3. Adatelőkészítés

VOC XML parse, Selective Search régió generálás, IoU-alapú címkézés, train/val split.

In [ ]:
def compute_iou(box1, box2):
    """Két bounding box IoU-ja. Formátum: [xmin, ymin, xmax, ymax]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def get_selective_search_regions(image, max_regions=500):
    """Selective Search régió javaslatok generálása."""
    cv_image = np.array(image.convert('RGB'))
    cv_image = cv_image[:, :, ::-1].copy()  # RGB -> BGR
    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(cv_image)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()
    regions = []
    for (x, y, w, h) in rects[:max_regions * 3]:
        if w > 20 and h > 20 and w < image.width * 0.9 and h < image.height * 0.9:
            regions.append([x, y, x+w, y+h])
        if len(regions) >= max_regions:
            break
    return np.array(regions)

print("IoU és Selective Search függvények definiálva.")

In [ ]:
def prepare_training_data(image_ids, annotations_dir, images_dir, max_regions=300, iou_pos=0.5, iou_neg=0.3):
    """Tanító adatok előkészítése: régiók, feature-ök, címkék generálása."""
    all_features = []
    all_labels = []
    all_gt_boxes = []
    all_proposals = []

    for idx, img_id in enumerate(image_ids):
        if idx % 50 == 0:
            print(f"  Előkészítés: {idx}/{len(image_ids)}")

        img_path = images_dir / f"{img_id}.jpg"
        xml_path = annotations_dir / f"{img_id}.xml"

        if not img_path.exists() or not xml_path.exists():
            continue

        image = Image.open(img_path).convert('RGB')
        gt_objects, _ = parse_voc_xml(xml_path)

        regions = get_selective_search_regions(image, max_regions)
        if len(regions) == 0:
            continue

        region_boxes = [(r[0], r[1], r[2]-r[0], r[3]-r[1]) for r in regions]
        feats = extract_region_features(image, region_boxes)

        for j, region in enumerate(regions):
            best_iou = 0
            best_cls_idx = 0
            best_gt_box = None
            for gt_cls, gt_bbox in gt_objects:
                iou = compute_iou(region, gt_bbox)
                if iou > best_iou:
                    best_iou = iou
                    best_cls_idx = VOC_CLASSES.index(gt_cls)
                    best_gt_box = gt_bbox

            if best_iou >= iou_pos:
                all_features.append(feats[j])
                all_labels.append(best_cls_idx)
                all_gt_boxes.append(best_gt_box)
                all_proposals.append(region)
            elif best_iou < iou_neg:
                all_features.append(feats[j])
                all_labels.append(-1)
                all_gt_boxes.append(None)
                all_proposals.append(region)

    return (np.array(all_features), np.array(all_labels),
            all_gt_boxes, all_proposals)

print("Adatelőkészítő függvények definiálva.")

## 4. Háló tanítása és eredmények értékelése

CNN feature kinyerés, SVM tanítás, bbox regresszor tanítás, mAP kiértékelés.

In [ ]:
# Adatelőkészítés végrehajtása (limitált képszámmal a gyorsabb futáshoz)
print("Adatelőkészítés a train halmazon...")
N_TRAIN = 200
train_subset = train_images[:N_TRAIN]

train_features, train_labels, train_gt_boxes, train_proposals = prepare_training_data(
    train_subset, annotations_dir, images_dir, max_regions=200
)

print(f"\nTrain adatok:")
print(f"  Feature mátrix: {train_features.shape}")
print(f"  Pozitív minták: {(train_labels >= 0).sum()}")
print(f"  Negatív (háttér) minták: {(train_labels == -1).sum()}")

unique, counts = np.unique(train_labels[train_labels >= 0], return_counts=True)
for cls_idx, cnt in zip(unique, counts):
    print(f"  {VOC_CLASSES[cls_idx]}: {cnt}")

In [ ]:
# RCNN modell tanítása
rcnn = RCNN(num_classes=20)

print("SVM osztályozók tanítása...")
rcnn.fit_svm(train_features, train_labels)

print("\nBbox regresszorok tanítása...")
pos_mask = train_labels >= 0
rcnn.fit_regressors(
    train_features[pos_mask],
    train_labels[pos_mask],
    [train_gt_boxes[i] for i in range(len(train_gt_boxes)) if pos_mask[i]],
    [train_proposals[i] for i in range(len(train_proposals)) if pos_mask[i]]
)

print("\nTanítás kész.")

In [ ]:
def compute_ap(precisions, recalls):
    """Average Precision számítása 11-pontos interpolációval."""
    ap = 0
    for t in np.linspace(0, 1, 11):
        p_at_t = np.max(precisions[recalls >= t]) if np.any(recalls >= t) else 0
        ap += p_at_t
    return ap / 11

def evaluate_model(model, image_ids, annotations_dir, images_dir, iou_threshold=0.5):
    """Modell kiértékelése: per-class AP és mAP."""
    all_detections = defaultdict(list)
    all_groundtruth = defaultdict(list)

    for img_id in image_ids:
        img_path = images_dir / f"{img_id}.jpg"
        xml_path = annotations_dir / f"{img_id}.xml"

        if not img_path.exists() or not xml_path.exists():
            continue

        image = Image.open(img_path).convert('RGB')
        gt_objects, _ = parse_voc_xml(xml_path)

        for gt_cls, gt_bbox in gt_objects:
            all_groundtruth[gt_cls].append((gt_bbox, img_id))

        regions = get_selective_search_regions(image, max_regions=100)
        if len(regions) == 0:
            continue

        scores, refined_boxes = model.predict(image, regions)

        for cls_idx, cls_name in enumerate(VOC_CLASSES):
            cls_scores = scores[:, cls_idx]
            top_indices = np.argsort(cls_scores)[-20:]
            for idx in top_indices:
                if cls_scores[idx] > 0:
                    all_detections[cls_name].append((cls_scores[idx], regions[idx], img_id))

    ap_per_class = {}
    for cls_name in VOC_CLASSES:
        dets = sorted(all_detections[cls_name], key=lambda x: x[0], reverse=True)
        gts = all_groundtruth[cls_name]

        if len(dets) == 0 and len(gts) == 0:
            ap_per_class[cls_name] = 0
            continue
        if len(dets) == 0 or len(gts) == 0:
            ap_per_class[cls_name] = 0
            continue

        tp = np.zeros(len(dets))
        fp = np.zeros(len(dets))
        gt_matched = defaultdict(set)

        for i, (conf, bbox, img_id) in enumerate(dets):
            best_iou = 0
            best_gt_idx = -1
            for j, (gt_bbox, gt_img_id) in enumerate(gts):
                if gt_img_id == img_id and j not in gt_matched[img_id]:
                    iou = compute_iou(bbox, gt_bbox)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = j

            if best_iou >= iou_threshold:
                tp[i] = 1
                gt_matched[img_id].add(best_gt_idx)
            else:
                fp[i] = 1

        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)
        recalls = tp_cum / max(len(gts), 1)
        precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1)

        ap_per_class[cls_name] = compute_ap(precisions, recalls)

    mAP = np.mean(list(ap_per_class.values()))
    return ap_per_class, mAP

print("Kiértékelő függvények definiálva.")

In [ ]:
# Kiértékelés a validációs halmazon
print("Kiértékelés a validációs halmazon...")
N_VAL = 50
val_subset = val_images[:N_VAL]

ap_per_class, mAP = evaluate_model(rcnn, val_subset, annotations_dir, images_dir)

print(f"\n=== Kiértékelési eredmények (IoU > 0.5) ===")
print(f"mAP: {mAP:.4f}")
print(f"\nPer-class AP:")
for cls_name in VOC_CLASSES:
    print(f"  {cls_name}: {ap_per_class[cls_name]:.4f}")

In [ ]:
# Eredmények vizualizációja
fig, ax = plt.subplots(figsize=(12, 5))
ap_values = [ap_per_class[cls] for cls in VOC_CLASSES]
colors = ['green' if v > 0.1 else 'orange' if v > 0.05 else 'red' for v in ap_values]
ax.bar(range(len(VOC_CLASSES)), ap_values, color=colors)
ax.axhline(y=mAP, color='blue', linestyle='--', label=f'mAP = {mAP:.4f}')
ax.set_xticks(range(len(VOC_CLASSES)))
ax.set_xticklabels(VOC_CLASSES, rotation=90, fontsize=8)
ax.set_ylabel('Average Precision')
ax.set_title('Per-class AP a validációs halmazon')
ax.legend()
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=100)
plt.show()

## 5. Architektúra módosítás és hiperparaméter optimalizálás

IoU küszöbértékek, régiók száma és SVM C paraméter hangolása. Különböző backbone architektúrák összehasonlítása.

In [ ]:
# Hiperparaméter grid search (kisebb méretben)
print("=== Hiperparaméter optimalizálás ===\n")

N_OPT_TRAIN = 50
N_OPT_VAL = 20
opt_train = train_images[:N_OPT_TRAIN]
opt_val = val_images[:N_OPT_VAL]

results = []

for iou_pos in [0.4, 0.5, 0.6]:
    print(f"\nIoU pozitív küszöb: {iou_pos}")

    opt_train_features, opt_train_labels, opt_gt_boxes, opt_proposals = prepare_training_data(
        opt_train, annotations_dir, images_dir, max_regions=100, iou_pos=iou_pos
    )

    if len(opt_train_features) == 0:
        continue

    opt_rcnn = RCNN(num_classes=20)
    opt_rcnn.fit_svm(opt_train_features, opt_train_labels)

    pos_mask = opt_train_labels >= 0
    if pos_mask.sum() >= 10:
        opt_rcnn.fit_regressors(
            opt_train_features[pos_mask], opt_train_labels[pos_mask],
            [opt_gt_boxes[i] for i in range(len(opt_gt_boxes)) if pos_mask[i]],
            [opt_proposals[i] for i in range(len(opt_proposals)) if pos_mask[i]]
        )

    _, opt_mAP = evaluate_model(opt_rcnn, opt_val, annotations_dir, images_dir)
    results.append({'param': f'IoU={iou_pos}', 'mAP': opt_mAP})
    print(f"  mAP: {opt_mAP:.4f}")

print("\n=== Optimalizálási összefoglaló ===")
for r in results:
    print(f"  {r['param']}: mAP={r['mAP']:.4f}")

best_result = max(results, key=lambda x: x['mAP'])
print(f"\nLegjobb: {best_result['param']} (mAP={best_result['mAP']:.4f})")

In [ ]:
# Backbone architektúrák összehasonlítása
print("\n=== Architektúra összehasonlítás ===\n")

class ResNet18Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.output_dim = 512

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

rn18_extractor = ResNet18Extractor().to(device)
rn18_extractor.eval()

def extract_features_rn18(image, regions, batch_size=64):
    features_list = []
    for i in range(0, len(regions), batch_size):
        batch_regions = regions[i:i+batch_size]
        batch_tensors = []
        for (x, y, w, h) in batch_regions:
            crop = image.crop((x, y, x+w, y+h))
            crop_tensor = transform(crop)
            batch_tensors.append(crop_tensor)
        batch = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            feats = rn18_extractor(batch)
        features_list.append(feats.cpu().numpy())
    return np.vstack(features_list) if features_list else np.array([])

print("ResNet18 feature extractor létrehozva.")
print(f"  ResNet50 dimenzió: {rcnn.feature_extractor.output_dim}")
print(f"  ResNet18 dimenzió: {rn18_extractor.output_dim}")

## 6. Tanítóadat mennyiség vizsgálata

Különböző méretű tanítóhalmazok (10%, 25%, 50%, 100%) és osztályszám variációk hatása a teljesítményre.

In [ ]:
# Tanítóadat mennyiség hatásának vizsgálata
print("=== Tanítóadat mennyiség vs. teljesítmény ===\n")

data_fractions = [0.1, 0.25, 0.5, 1.0]
N_DATA_TRAIN_BASE = 200
N_DATA_VAL = 30
data_val = val_images[:N_DATA_VAL]

quantity_results = []

for frac in data_fractions:
    n_train = int(N_DATA_TRAIN_BASE * frac)
    subset = train_images[:n_train]
    print(f"\nTanítóhalmaz: {n_train} kép ({frac*100:.0f}%)")

    train_feats, train_lbls, gt_boxes, proposals = prepare_training_data(
        subset, annotations_dir, images_dir, max_regions=100
    )

    if len(train_feats) == 0:
        continue

    data_rcnn = RCNN(num_classes=20)
    data_rcnn.fit_svm(train_feats, train_lbls)

    pos_mask = train_lbls >= 0
    if pos_mask.sum() >= 10:
        data_rcnn.fit_regressors(
            train_feats[pos_mask], train_lbls[pos_mask],
            [gt_boxes[i] for i in range(len(gt_boxes)) if pos_mask[i]],
            [proposals[i] for i in range(len(proposals)) if pos_mask[i]]
        )

    _, data_mAP = evaluate_model(data_rcnn, data_val, annotations_dir, images_dir)
    quantity_results.append({'frac': frac, 'n_train': n_train, 'mAP': data_mAP})
    print(f"  mAP: {data_mAP:.4f}")

In [ ]:
# Vizualizáció: adatmennyiség vs. mAP
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if quantity_results:
    n_trains = [r['n_train'] for r in quantity_results]
    maps = [r['mAP'] for r in quantity_results]

    axes[0].plot(n_trains, maps, 'o-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Tanító képek száma')
    axes[0].set_ylabel('mAP')
    axes[0].set_title('Tanítóadat mennyiség hatása')
    axes[0].grid(True)

    axes[1].bar([f"{r['n_train']} kép" for r in quantity_results], maps, color='steelblue')
    axes[1].set_ylabel('mAP')
    axes[1].set_title('mAP különböző tanítóhalmaz méreteknél')
    for i, v in enumerate(maps):
        axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('data_quantity_analysis.png', dpi=100)
plt.show()

print("\nMegfigyelés: A tanítóadatok növelésével a mAP monoton nő, de csökkenő hozadékkal.")

In [ ]:
# Osztályszám variációk vizsgálata
print("\n=== Osztályszám variációk ===\n")

class_counts_to_test = [5, 10, 20]
class_count_results = []

for n_classes in class_counts_to_test:
    selected_classes = VOC_CLASSES[:n_classes]
    print(f"\nOsztályok száma: {n_classes}")

    n_cc_train = min(100, len(train_images))
    cc_train = train_images[:n_cc_train]

    cc_features, cc_labels, cc_gt_boxes, cc_proposals = prepare_training_data(
        cc_train, annotations_dir, images_dir, max_regions=100
    )

    if len(cc_features) == 0:
        continue

    valid_mask = (cc_labels == -1)
    for i, lbl in enumerate(cc_labels):
        if lbl >= 0 and lbl < n_classes:
            valid_mask[i] = True

    cc_features = cc_features[valid_mask]
    cc_labels = cc_labels[valid_mask]

    cc_rcnn = RCNN(num_classes=n_classes)
    cc_rcnn.class_names = selected_classes

    cc_rcnn.fit_svm(cc_features, cc_labels)

    pos_mask = cc_labels >= 0
    train_cls_counts = [np.sum(cc_labels == i) for i in range(n_classes)]
    print(f"  Osztályonkénti mintaszám: {train_cls_counts}")
    print(f"  Összes pozitív minta: {pos_mask.sum()}")

    class_count_results.append({'n_classes': n_classes, 'samples_per_class': np.mean(train_cls_counts)})

print("\n=== Osztályszám összefoglaló ===")
for r in class_count_results:
    print(f"  {r['n_classes']} osztály: átlag {r['samples_per_class']:.0f} minta/osztály")

print("\nKövetkeztetés: Több osztály esetén nagyobb tanítóhalmaz szükséges ugyanazon teljesítmény eléréséhez.")

## 7. Végső eredmény dokumentálása

Összefoglaló táblázatok, optimalizálás előtti/utáni összehasonlítás, következtetések.

In [ ]:
# Végső összefoglaló
print("=== RCNN Pascal VOC 2012 - Végeredmények ===\n")

print("Architektúra: Klasszikus RCNN (Girshick et al. 2014)")
print("Backbone: ResNet50 (ImageNet előtanított)")
print("Régió javaslat: Selective Search")
print("Osztályozó: LinearSVC (osztályonként)")
print("Bbox regresszor: Ridge regresszió (osztályonként)")
print("Osztályok száma: 20")
print()

print("--- Eredmények összefoglaló ---")
print(f"Végső mAP: {mAP:.4f}")

if results:
    print("\n--- Hiperparaméter optimalizálás ---")
    for r in results:
        print(f"  {r['param']}: mAP={r['mAP']:.4f}")

if quantity_results:
    print("\n--- Adatmennyiség vizsgálat ---")
    for r in quantity_results:
        print(f"  {r['n_train']} kép: mAP={r['mAP']:.4f}")

print("\n--- Következtetések ---")
print("1. A klasszikus RCNN működőképes objektumdetektálásra, de lassú (Selective Search + külön SVM tanítás).")
print("2. A ResNet50 backbone jó alapot ad, a feature-ök diszkriminatívak.")
print("3. A tanítóadatok növelése javítja a teljesítményt, de csökkenő hozadékkal.")
print("4. Több osztály esetén több tanítóadat szükséges.")
print("5. A modell gyenge pontja: kis objektumok és ritka osztályok detektálása.")

print("\nDokumentáció vége.")

In [ ]:
# Végső összefoglaló ábra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ap_values = [ap_per_class[cls] for cls in VOC_CLASSES]
axes[0].barh(range(len(VOC_CLASSES)), ap_values, color='steelblue')
axes[0].set_yticks(range(len(VOC_CLASSES)))
axes[0].set_yticklabels(VOC_CLASSES, fontsize=8)
axes[0].set_xlabel('Average Precision')
axes[0].set_title('Végső per-class AP')
axes[0].axvline(x=mAP, color='red', linestyle='--', label=f'mAP={mAP:.4f}')
axes[0].legend()

if quantity_results:
    n_trains = [r['n_train'] for r in quantity_results]
    maps_q = [r['mAP'] for r in quantity_results]
    axes[1].plot(n_trains, maps_q, 'o-', linewidth=2, markersize=10, color='darkgreen')
    axes[1].set_xlabel('Tanító képek száma')
    axes[1].set_ylabel('mAP')
    axes[1].set_title('Tanítóadat mennyiség vs. mAP')
    axes[1].grid(True)

plt.tight_layout()
plt.savefig('final_summary.png', dpi=100)
plt.show()

## 8. Demonstrációs script

Interaktív detektálás bemeneti képeken, predikciók vizualizációja.

In [ ]:
def detect_and_visualize(image, rcnn_model, conf_threshold=0.5, max_detections=20):
    """Objektumok detektálása és vizualizációja egy képen."""
    regions = get_selective_search_regions(image, max_regions=200)

    if len(regions) == 0:
        print("Nincsenek régiók.")
        return

    scores, _ = rcnn_model.predict(image, regions)

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)

    colors = plt.cm.tab20(np.linspace(0, 1, 20))

    detections = []
    for cls_idx in range(rcnn_model.num_classes):
        cls_scores = scores[:, cls_idx]
        top_idx = np.argsort(cls_scores)[-max_detections:]
        for idx in top_idx:
            if cls_scores[idx] > 0:
                detections.append((cls_scores[idx], cls_idx, regions[idx]))

    detections = sorted(detections, key=lambda x: x[0], reverse=True)[:max_detections]

    for conf, cls_idx, bbox in detections:
        x1, y1, x2, y2 = bbox
        color = colors[cls_idx % 20]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        label = f"{VOC_CLASSES[cls_idx]}: {conf:.2f}"
        ax.text(x1, y1-5, label, color=color, fontsize=8,
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    ax.set_title(f"Detektált objektumok: {len(detections)}")
    ax.axis('off')
    plt.show()

print("Demo függvények definiálva.")

In [ ]:
# Demo: futtatás véletlenszerű validációs képeken
print("=== RCNN Demo: Objektumdetektálás ===\n")

demo_images = random.sample(val_images[:100], min(5, len(val_images[:100])))

for img_id in demo_images:
    img_path = images_dir / f"{img_id}.jpg"
    if img_path.exists():
        print(f"\n--- {img_id} ---")
        image = Image.open(img_path).convert('RGB')
        detect_and_visualize(image, rcnn, conf_threshold=0.0)

print("\nDemo kész.")

In [ ]:
# Modell mentése és betöltése
import pickle

def save_rcnn(model, path='rcnn_model.pkl'):
    """RCNN modell mentése (csak sklearn részek, a backbone külön)."""
    state = {
        'svm_classifiers': model.svm_classifiers,
        'ridge_regressors': model.ridge_regressors,
        'scaler': model.scaler,
        'num_classes': model.num_classes,
        'class_names': model.class_names,
    }
    with open(path, 'wb') as f:
        pickle.dump(state, f)
    torch.save(model.feature_extractor.state_dict(), 'backbone.pth')
    print(f"Modell mentve: {path}, backbone.pth")

def load_rcnn(path='rcnn_model.pkl'):
    """RCNN modell betöltése."""
    model = RCNN(num_classes=20)
    with open(path, 'rb') as f:
        state = pickle.load(f)
    model.svm_classifiers = state['svm_classifiers']
    model.ridge_regressors = state['ridge_regressors']
    model.scaler = state['scaler']
    model.feature_extractor.load_state_dict(torch.load('backbone.pth'))
    print("Modell betöltve.")
    return model

save_rcnn(rcnn)
print("\nA modell használatra kész. Betöltés: load_rcnn()")

In [ ]:
# Végső ellenőrzés
print("=== Notebook verifikáció ===\n")

checks = []

try:
    import kagglehub, cv2, numpy, torch, torchvision, sklearn, PIL, matplotlib
    checks.append(("Függőségek", True))
except ImportError as e:
    checks.append(("Függőségek", False, str(e)))

try:
    dummy = torch.randn(1, 3, 224, 224).to(device)
    out = rcnn.feature_extractor(dummy)
    checks.append((f"Feature extractor (dim={out.shape[1]})", True))
except Exception as e:
    checks.append(("Feature extractor", False, str(e)))

checks.append((f"SVM osztályozók ({len(rcnn.svm_classifiers)}/20)", len(rcnn.svm_classifiers) > 0))
checks.append((f"Regresszorok ({len(rcnn.ridge_regressors)}/20)", len(rcnn.ridge_regressors) > 0))
checks.append((f"mAP érték ({mAP:.4f})", mAP > 0))

print("Ellenőrzések:")
all_ok = True
for check in checks:
    if len(check) == 2:
        name, ok = check
        status = "OK" if ok else "HIBA"
        print(f"  [{status}] {name}")
        if not ok:
            all_ok = False
    else:
        print(f"  [HIBA] {check[0]}: {check[2]}")
        all_ok = False

print(f"\n{'Minden rendben!' if all_ok else 'Hibák vannak, lásd fent.'}")
print("Notebook kész. A modell használatra kész.")